#Notebook feito por Ian Marques Breda
#Windows 11 home - python 3.12.5

In [1]:
#Bibliotecas necessárias

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
def extra_predict(acuracias):
# 1. Modelo com log(b * x)
    def modelo_log_bx(x, a, b, c):
        return (a*b)/(b+x) + c

    # 2. Dados (todos x > 0)
    x_dados = np.array([1, 2, 3, 4, 5, 6, 7])
    y_dados = np.array(acuracias)

    # 3. Ajuste do modelo
    # Fornecendo estimativas iniciais para os parâmetros (necessário para estabilidade)
    param_iniciais = [1.0, 1.0, 1.0]  # a, b, c
    param, cov = curve_fit(modelo_log_bx, x_dados, y_dados, p0=param_iniciais, maxfev=10000)

    # 4. Extrapolação até x = 100
    x_pred = np.linspace(1, 20, 200)
    y_pred = modelo_log_bx(x_pred, *param)

    # 5. Plotagem
    # plt.figure(figsize=(10, 6))
    # plt.scatter(x_dados, y_dados, label="Dados originais", color="blue")
    # plt.plot(x_pred, y_pred, label="Ajuste logarítmico (a⋅log(b⋅x)+c)", color="green")
    # plt.xlabel("x")
    # plt.ylabel("y")
    # plt.title("Extrapolação com Modelo Logarítmico: y = a⋅log(b⋅x)+c")
    # plt.legend()
    # plt.grid(True)
    # plt.show()

    # 6. Imprimir os parâmetros ajustados
    # print(f"Parâmetros ajustados:\na = {param[0]:.4f}, b = {param[1]:.4f}, c = {param[2]:.4f}")
    # print(np.mean(y_pred))

    return 100 * np.mean(y_pred)


In [2]:
#Transformação dos dados

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

#Download do dataset
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform)

#parâmetros de batch, épocas e proporção do conjunto de validação
batch = 30
num_epochs = 20
val_ratio = 0.1

#Divisão dos conjuntos e criação dos dataloaders
val_size = int(len(train_data) * val_ratio)
train_size = len(train_data) - val_size

train_subset, val_subset = random_split(train_data, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=batch, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch, shuffle=False)

test_loader = DataLoader(test_data, batch_size=batch, shuffle=False)

print('Dados carregados \n')

Dados carregados 



In [3]:
class SimpleDynamicCNN(nn.Module):
    def __init__(self, kernel, qtd_seq, type_pooling, neuron, func, num_convs, 
                 num_fcs, input_shape=(3, 32, 32), num_classes=10):
        super(SimpleDynamicCNN, self).__init__()

        # Criar camadas convolucionais
        conv_layers = []
        in_channels = input_shape[0]
        out_channels = 32
        pad = 1 if kernel == 3 else 2

        for _ in range(qtd_seq):
            for _ in range(num_convs):
                conv_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=kernel, padding=pad))
                
                if func == 'elu':
                    conv_layers.append(nn.ELU())
                else:
                    conv_layers.append(nn.LeakyReLU())
                in_channels = out_channels

            if type_pooling == 'avg':
                conv_layers.append(nn.AvgPool2d(kernel_size=2, stride=2))
            else:
                conv_layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            out_channels *= 2
        conv_layers.append(nn.Flatten())


        self.conv_net = nn.Sequential(*conv_layers)

        # Estimar tamanho do flatten depois das convoluções
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_shape)
            x = self.conv_net(dummy_input)
            flattened_size = x.view(1, -1).size(1)

        # Criar camadas totalmente conectadas
        fc_dims = [flattened_size]
        arch = [10, 100, 20, 90, 30, 80, 40, 70, 50, 60]

        if neuron == 'fix':
            for _ in range(num_fcs - 1):
                fc_dims.append(max(32, fc_dims[-1] // 2))
            fc_dims.append(num_classes)
        else:
            rand = np.random.choice(arch, size=num_fcs)
            rand = rand.tolist()
            fc_dims = fc_dims + rand
            fc_dims.append(num_classes)

        fc_layers = []
        for i in range(len(fc_dims) - 1):
            fc_layers.append(nn.Linear(fc_dims[i], fc_dims[i + 1]))
            if i < len(fc_dims) - 2:
                fc_layers.append(nn.ReLU())

        self.fc_net = nn.Sequential(*fc_layers)

    def forward(self, x):
        x = self.conv_net(x)
        x = self.fc_net(x)
        return x

In [4]:
#Early stop para evitar overfitting
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.01):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [5]:
device = 'cuda'

def experiment(otim, kernel, qtd_seq, type_pooling, neuron, func, num_epochs, num_convs, num_fcs):

    #Salva os valores de loss do treinamento e validação para plotagem
    train_losses = []
    val_losses = []

    #Seleciona a arquitetura do modelo
    model = SimpleDynamicCNN(kernel, qtd_seq, type_pooling, neuron, func, num_convs, 
                             num_fcs, input_shape=(3, 32, 32), num_classes=10)
    model.to(device)
    
    #Função de perda
    criterion = nn.CrossEntropyLoss()

    #Seleciona o otimizador
    if otim == 'sgd':
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

    early_stopping = EarlyStopping(patience=3, min_delta=0.01)
    n_total_steps = len(train_loader)

    for epoch in range(num_epochs):

        #Coloca o modelo em modo de treino e inicia o treinamento
        model.train()
        running_loss = 0.0

        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward e otimização dos pesos
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            running_loss += loss.item()
        avg_train_loss = running_loss / len(train_loader)

        
        #Coloca o modelo em modo de validação
        model.eval()
        val_loss = 0.0
        n_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                n_correct += (predicted == labels).sum().item()

        acc = 100.0 * n_correct / len(val_loader.dataset)
        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        #Print da época, e loss de treino/teste
        #print(f'Época [{epoch+1}/{num_epochs}], Treinamento Loss: {avg_train_loss:.4f}, Validação Loss: {avg_val_loss:.4f}')
        #print(f'Acuracia do modelo (Validação): {acc}%\n')

        #No caso de early stop, salva o modelo utilizado para futura utilização
        early_stopping(avg_val_loss)
        if early_stopping.early_stop:
            #print("Early stopping ativado!")
            break

    #Acurácia do modelo no conjunto de testes
    with torch.no_grad():
        model.eval()
        n_correct = 0
        n_samples = len(test_loader.dataset)

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)

            _, predicted = torch.max(outputs, 1)
            n_correct += (predicted == labels).sum().item()

        acc = 100.0 * n_correct / n_samples
        #print(f'Acuracia do modelo (Testes): {acc}%')

        with open("resultados.txt", "a") as f:
            f.write(f'Acuracia do modelo (Testes): {acc}%\n\n')
    
    #Plotagem da loss de treino e validação
    # plt.figure(figsize=(8, 5))
    # plt.plot(train_losses, label='Train Loss')
    # plt.plot(val_losses, label='Val Loss')
    # plt.xlabel('Epoch')
    # plt.ylabel('Loss')
    # plt.title(f'Convoluções: {num_convs} | FC: {num_fcs} | Otimizador: {otim.upper()}')
    # plt.legend()
    # plt.grid(True)
    # plt.tight_layout()
    # plt.show()

    return acc

In [6]:

def roleta(qtd_ind, apt):
    return np.random.choice(range(qtd_ind), p=apt, size=qtd_ind)

# Função de crossover
def crossover(individuos, selecao, ponto, cross, t_cross):
    indice = 0
    n_ind = []
    for p in cross:
        pai_a = individuos[selecao[indice]]
        pai_b = individuos[selecao[indice+1]]
        filho_a = []
        filho_b = []
        if p <= t_cross:
            filho_a = np.concatenate((pai_a[:ponto], pai_b[ponto:]))
            filho_b = np.concatenate((pai_b[:ponto], pai_a[ponto:]))
        else:
            filho_a = pai_a.copy()
            filho_b = pai_b.copy()
        n_ind.append(filho_a)
        n_ind.append(filho_b)
        indice += 2
    return n_ind

# Função de mutação
def mutacao(new_individuos, tam_alvo, qtd_ind, t_mut):
    for j in range(qtd_ind):
        for i in range(tam_alvo):
            p = np.random.uniform(0, 1)
            if p <= t_mut:
                new_individuos[j][i] = 1 if new_individuos[j][i] == 0 else 0
    return new_individuos

In [7]:
def lyapunov_exponent(r, x0=0.5, n_iter=1000, n_trans=100):
    x = x0
    lyapunov_sum = 0.0
    for i in range(n_iter):
        x = r * x * (1 - x)
        if i >= n_trans:
            lyapunov_sum += np.log(abs(r * (1 - 2 * x)))
    return lyapunov_sum / (n_iter - n_trans)


In [8]:
modelos = np.random.randint(0, 2, (20, 10))
evo_best_apt = []
geracoes = 10
r = np.random.uniform(3.5, 3.7)

t_cross = np.random.uniform(0, 1)
t_mut = np.random.uniform(0, 1)

for g in range(geracoes):
    apt = []
    for j in range(20):
        num_convs = 0
        num_fcs = 0
        mult = 2

        for i in range(6, 8):
            num_convs += modelos[j][i]*mult
            num_fcs += modelos[j][i+2]*mult
            mult /= 2

        num_convs += 1
        num_fcs += 1 

        otim = 'adam' if modelos[j][0] == 0 else 'sgd'
        kernel = 3 if modelos[j][1] == 0 else 5
        qtd_seq = modelos[j][2] + 2
        type_pooling = 'max' if modelos[j][3] == 0 else 'avg'
        neuron = 'fix' if modelos[j][4] == 0 else 'rand'
        func = 'elu' if modelos[j][5] == 0 else 'leaky'

        with open("resultados.txt", "a") as f:
            f.write(
                    f"\nModelo {j+1} | Geração {g}\n"
                    f"Otimizador: {otim.upper()} | Filtro: {kernel} | Camadas de convolução: {qtd_seq}\n"
                    f"Pooling: {type_pooling.upper()} | Neurônios: {neuron.upper()} | Ativação: {func.upper()}\n"
                    f"Convs por camada: {num_convs} | FC ocultas: {num_fcs}\n"
            )

        acuracia = experiment(otim, kernel, qtd_seq, type_pooling, neuron, func, num_epochs, int(num_convs), int(num_fcs))
        apt.append(acuracia)
        melhor_index = np.argmax(apt)
        with open("resultados.txt", "a") as f:
            f.write(f"\n\n==============MELHOR ACURÁCIA DA GERAÇÃO: {apt[melhor_index]}=================\n\n")

    soma_aptidao = sum(apt)
    apt_normalizada = [x/soma_aptidao for x in apt]
    selecao = roleta(20, apt_normalizada)
    cross = np.random.uniform(0, 1, 10)
    ponto = np.random.randint(1, 9)
    modelos = crossover(modelos, selecao, ponto, cross, t_cross)
    modelos = mutacao(modelos, 10, 20, t_mut)

    t_cross = t_cross * r * (1 - t_cross)
    t_mut = t_mut * r * (1 - t_mut)

    # Calcula o expoente de Lyapunov
    lambda_r = lyapunov_exponent(r, x0=t_cross)
    print(f'Geração {g}, r utilizado: {r}, lambda (lyapunov): {lambda_r}')
    print(f'Taxa de crossover: {t_cross}, taxa de mutação: {t_mut}\n')
    # Faixa desejada de caos
    if g < 5:
        lyap_min = 0.3
        lyap_max = 0.5
    else:
        lyap_min = 0.0
        lyap_max = 0.2

    # Ajusta r dinamicamente
    if lambda_r < lyap_min:
        r += 0.05  # Aumenta o caos
    elif lambda_r > lyap_max:
        r -= 0.05  # Diminui o caos

    # Mantém r dentro dos limites válidos do mapa logístico
    r = min(max(r, 3.5), 3.7)
    

Geração 0, r utilizado: 3.5062500819982327, lambda (lyapunov): -0.4522050858357858
Taxa de crossover: 0.5364014318118009, taxa de mutação: 0.32836500047512834

Geração 1, r utilizado: 3.5562500819982326, lambda (lyapunov): -0.22745516389166198
Taxa de crossover: 0.8843502606946985, taxa de mutação: 0.7843004676326157

Geração 2, r utilizado: 3.6062500819982324, lambda (lyapunov): -0.0049961247570514565
Taxa de crossover: 0.368828783942361, taxa de mutação: 0.6100810254215099

Geração 3, r utilizado: 3.656250081998232, lambda (lyapunov): 0.26846497986738427
Taxa de crossover: 0.8511534913737477, taxa de mutação: 0.8697566956787742

Geração 4, r utilizado: 3.7, lambda (lyapunov): 0.35968138435336666
Taxa de crossover: 0.46875753433530093, taxa de mutação: 0.4191359482026431

Geração 5, r utilizado: 3.7, lambda (lyapunov): 0.35922306727341613
Taxa de crossover: 0.9213884608550033, taxa de mutação: 0.9008057189695832

Geração 6, r utilizado: 3.6500000000000004, lambda (lyapunov): 0.2593248